In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Text, DateTime, ForeignKey, MetaData, Sequence, Index
from sqlalchemy.orm import sessionmaker, relationship, declarative_base

In [ ]:

Base = declarative_base()
exp_id = Sequence('exp_id')
param_id = Sequence('param_id')
trial_id = Sequence('trial_id')
event_id = Sequence('event_id')

In [ ]:
class Experiment(Base):
    __tablename__ = 'experiments'
    # id = Column('ExperimentID', Integer, primary_key=True, autoincrement=True)
    id = Column('ExperimentID', Integer, server_default=exp_id.next_value(), primary_key=True)
    subject_name = Column('SubjectName', String)
    session_number = Column('SessionNumber', Integer)
    experiment_type = Column('ExperimentType', String)
    comment = Column('Comment', Text)
    start_datetime = Column('StartDateTime', DateTime)
    log_file_name = Column('LogFileName', String)
    measurement_file_name = Column('MeasurementFileName', String)
    parameters = relationship('Parameter', back_populates='experiment')
    trials = relationship('Trial', back_populates='experiment')

class Parameter(Base):
    __tablename__ = 'parameters'
    id = Column('ParameterID', Integer, server_default=param_id.next_value(), primary_key=True)
    experiment_id = Column('ExperimentID', Integer, ForeignKey('experiments.ExperimentID'))
    parameter_name = Column('ParameterName', String)
    parameter_value = Column('ParameterValue', String)
    experiment = relationship('Experiment', back_populates='parameters')

class Trial(Base):
    __tablename__ = 'trials'
    id = Column('TrialID', Integer, server_default=trial_id.next_value(), primary_key=True)
    experiment_id = Column('ExperimentID', Integer, ForeignKey('experiments.ExperimentID'))
    trial_number = Column('TrialNumber', Integer)
    start_datetime = Column('StartDateTime', DateTime)
    events = relationship('Event', back_populates='trial')
    experiment = relationship('Experiment', back_populates='trials')

class Event(Base):
    __tablename__ = 'events'
    id = Column('EventID', Integer, server_default=event_id.next_value(), primary_key=True)
    trial_id = Column('TrialID', Integer, ForeignKey('trials.TrialID'))
    event_type = Column('EventType', String)
    event_datetime = Column('EventDateTime', DateTime)
    trial = relationship('Trial', back_populates='events')

In [ ]:

# Create an engine that stores data in the local directory's duckdb file.
engine = create_engine('duckdb:///experiment.ddb')

#metadata = MetaData(engine)

In [ ]:
# Create all tables in the engine.
Base.metadata.create_all(engine)

In [ ]:
# Create a configured "Session" class.
Session = sessionmaker(bind=engine)


In [ ]:

# Create a session.
session = Session()

In [ ]:
def add_experiment(subject_name, session_number, experiment_type, comment, start_datetime, log_file_name, measurement_file_name):
    new_experiment = Experiment(
        subject_name=subject_name,
        session_number=session_number,
        experiment_type=experiment_type,
        comment=comment,
        start_datetime=start_datetime,
        log_file_name=log_file_name,
        measurement_file_name=measurement_file_name
    )
    session.add(new_experiment)
    session.commit()

# Example method to query experiments.
def get_experiments():
    return session.query(Experiment).all()